In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.feature_selection import mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


In [3]:
url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"

df = pd.read_csv(url)

df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [4]:
print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

Shape: (5000, 9)

Columns:
['lead_source', 'industry', 'employment_status', 'location', 'annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score', 'converted']

Data types:
lead_source                  object
industry                     object
employment_status            object
location                     object
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

Missing values:
lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64


In [5]:
categorical_columns = df.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

numerical_columns = df.select_dtypes(
    include=["number"]
).columns.tolist()

print("Categorical columns:", categorical_columns)
print("Numerical columns:", numerical_columns)

Categorical columns: ['lead_source', 'industry', 'employment_status', 'location']
Numerical columns: ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score', 'converted']


In [6]:
df[categorical_columns] = df[categorical_columns].fillna("NA")
df[numerical_columns] = df[numerical_columns].fillna(0.0)

df.isnull().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

In [7]:
df["industry"].value_counts()

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

In [8]:
numerical_features = [
    col for col in numerical_columns
    if col != "converted"
]

correlation_matrix = df[numerical_features].corr()

correlation_matrix

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000000,0.161300,0.122842,0.229496
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204
interaction_count,0.122842,0.721609,1.000000,0.915746
lead_score,0.229496,0.757204,0.915746,1.000000


In [9]:
pairs = [
    ("interaction_count", "lead_score"),
    ("number_of_courses_viewed", "lead_score"),
    ("number_of_courses_viewed", "interaction_count"),
    ("annual_income", "interaction_count"),
]

for feature1, feature2 in pairs:
    print(
        feature1,
        "and",
        feature2,
        ":",
        correlation_matrix.loc[feature1, feature2]
    )

interaction_count and lead_score : 0.9157457980418697
number_of_courses_viewed and lead_score : 0.7572042392104886
number_of_courses_viewed and interaction_count : 0.7216090038937035
annual_income and interaction_count : 0.12284235135955959


In [10]:
y = df["converted"]
X = df.drop(columns=["converted"])

In [11]:
df_full_train, df_test = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

df_train, df_val = train_test_split(
    df_full_train,
    test_size=0.25,
    random_state=42
)

y_train = df_train["converted"]
y_val = df_val["converted"]

X_train = df_train.drop(columns=["converted"])
X_val = df_val.drop(columns=["converted"])

X_test = df_test.drop(columns=["converted"])
y_test = df_test["converted"]

print("Training set:", X_train.shape)
print("Validation set:", X_val.shape)
print("Test set:", X_test.shape)

Training set: (3000, 8)
Validation set: (1000, 8)
Test set: (1000, 8)


In [15]:

categorical_features = X_train.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

# Convert categorical values into integer codes
X_cat = X_train[categorical_features].copy()

for col in categorical_features:
    X_cat[col] = X_cat[col].astype("category").cat.codes

# Calculate mutual information
mi_scores = mutual_info_classif(
    X_cat,
    y_train,
    discrete_features=True,
    random_state=42
)

# Display results rounded to 2 decimal places
mi_results = pd.Series(
    mi_scores,
    index=categorical_features
).sort_values(ascending=False)

mi_results.round(2)


lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64

In [16]:
categorical_features = X_train.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

numerical_features = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "numerical",
            "passthrough",
            numerical_features
        )
    ]
)

In [17]:
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                solver="liblinear",
                C=1.0,
                max_iter=1000,
                random_state=42
            )
        )
    ]
)

model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('categorical',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['lead_source', 'industry',
                                                   'employment_status',
                                                   'location']),
                                                 ('numerical', 'passthrough',
                                                  ['annual_income',
                                                   'number_of_courses_viewed',
                                                   'interaction_count',
                                                   'lead_score'])])),
                ('classifier',
                 LogisticRegression(max_iter=1000, random_state=42,
                                    solver='liblinear'))])

In [18]:
y_pred = model.predict(X_val)

accuracy = accuracy_score(y_val, y_pred)

round(accuracy, 2)

0.65

In [19]:
original_accuracy = accuracy_score(
    y_val,
    model.predict(X_val)
)

features_to_test = [
    "lead_source",
    "number_of_courses_viewed",
    "interaction_count"
]

results = {}

for feature in features_to_test:
    X_train_reduced = X_train.drop(columns=[feature])
    X_val_reduced = X_val.drop(columns=[feature])

    reduced_categorical = X_train_reduced.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()

    reduced_numerical = X_train_reduced.select_dtypes(
        include=["number"]
    ).columns.tolist()

    reduced_preprocessor = ColumnTransformer(
        transformers=[
            (
                "categorical",
                OneHotEncoder(handle_unknown="ignore"),
                reduced_categorical
            ),
            (
                "numerical",
                "passthrough",
                reduced_numerical
            )
        ]
    )

    reduced_model = Pipeline(
        steps=[
            ("preprocessor", reduced_preprocessor),
            (
                "classifier",
                LogisticRegression(
                    solver="liblinear",
                    C=1.0,
                    max_iter=1000,
                    random_state=42
                )
            )
        ]
    )

    reduced_model.fit(X_train_reduced, y_train)

    reduced_accuracy = accuracy_score(
        y_val,
        reduced_model.predict(X_val_reduced)
    )

    difference = original_accuracy - reduced_accuracy

    results[feature] = {
        "accuracy_without_feature": reduced_accuracy,
        "difference": difference
    }

pd.DataFrame(results).T

,accuracy_without_feature,difference
lead_source,0.642,0.003
number_of_courses_viewed,0.643,0.002
interaction_count,0.601,0.044


In [20]:
C_values = [0.000001, 0.00001, 0.0001, 0.001]

regularization_results = {}

for C in C_values:
    regularized_model = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            (
                "classifier",
                LogisticRegression(
                    solver="liblinear",
                    C=C,
                    max_iter=1000,
                    random_state=42
                )
            )
        ]
    )

    regularized_model.fit(X_train, y_train)

    predictions = regularized_model.predict(X_val)

    score = accuracy_score(y_val, predictions)

    regularization_results[C] = score

pd.Series(regularization_results).round(3)

0.000001    0.598
0.000010    0.598
0.000100    0.613
0.001000    0.645
dtype: float64

In [21]:
pd.DataFrame({
    "C": list(regularization_results.keys()),
    "validation_accuracy": [
        round(score, 3)
        for score in regularization_results.values()
    ]
})

,C,validation_accuracy
0,0.000001,0.598
1,0.000010,0.598
2,0.000100,0.613
3,0.001000,0.645
